In [ ]:
from PIL import Image
import PIL.ImageOps  
import numpy as np
import os
import cv2
import csv
Image.MAX_IMAGE_PIXELS = None
from utils import *
from models_inference import *

In [ ]:
raw_images_dir = "./raw_images/"
mass_channel_dir= raw_images_dir+'/mass_channel_images/'
membrane_channel_dir= raw_images_dir+'/membrane_channel_images/'
FR_removed_dir= "./FR_removed_RG_images/"
raw_segmented_masks_dir ="./raw_segmented_masks/"
individual_fibers_dir ='./individual_fibers/'
final_segmented_masks_dir= "./final_segmented_masks/"
model_FR_dir = "./model_FR/"
FR_model_name='FR_model'
model_segmentation_dir= "./model_fiber_segmentation/"
model_classification_dir= "./model_fiber_classification/"
c_limit_mass=0.01
c_limit_membrane= 0.01

In [ ]:
img_list = images_list(mass_channel_dir)
for infile in img_list:
    infile_mass_channel_path= os.path.join(mass_channel_dir, infile)
    infile_membrane_channel_path= os.path.join(membrane_channel_dir, infile)
    outfile_path= os.path.join(FR_removed_dir, infile)
    
    mass_img= cv2.imread(infile_mass_channel_path,0)
    membrane_img= cv2.imread(infile_membrane_channel_path,0)
    
    mass_contrast_img= contrast_stretching_adap(img_as_float(mass_img), clip_limit=c_limit_mass)
    membrane_contrast_img=contrast_stretching_adap(img_as_float(membrane_img), clip_limit=c_limit_membrane)
    
    red_green_img= make_merge_channels(membrane_contrast_img,mass_contrast_img)
    
    FR_model= get_unet_model(name=FR_model_name,directory= model_FR_dir)
    
    folded_region_img= unet_predict(red_green_img, FR_model)
    red_green_noFold= red_green_img[folded_region_img ==255] = 0
    
    red_green_noFold.save(outfile_path)
    
    


In [ ]:
img_list = images_list(FR_removed_dir)
for infile in img_list:
    infile_path= os.path.join(FR_removed_dir, infile)
    outfile_path= os.path.join(raw_segmented_masks_dir, infile)
    
    mass_img= cv2.imread(infile_mass_channel_path)
 
    
    mass_contrast_img= contrast_stretching_adap(img_as_float(mass_img), clip_limit=c_limit_mass)
    membrane_contrast_img=contrast_stretching_adap(img_as_float(membrane_img), clip_limit=c_limit_membrane)
    
    red_green_img= make_merge_channels(membrane_contrast_img,mass_contrast_img)
    
    FR_model= get_unet_model(name=FR_model_name,directory= model_FR_dir)
    
    folded_region_img= unet_predict(red_green_img, FR_model)
    red_green_noFold= red_green_img[folded_region_img ==255] = 0
    
    red_green_noFold.save(outfile_path)